<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Case study: tag research papers with many labels

**[Case study: tag research papers with many labels](https://learning.nextia-ai.com/courses/text-routing/m10/tag-research-papers-with-many-labels/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 10, case study 4 of 4 (optional) · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** take 12,000 real paper abstracts from arXiv, with all the categories that their authors chose, to a decision for an assistant that suggests categories to a person: one yes/no model per category, the long tail of rare categories, two levels (area, then category), thresholds chosen from a cost table, a fair comparison with embeddings and a recorded LLM, one test, and a run on new abstracts.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page. The notebook runs in about 2 to 3 minutes after the install. |
| **Needs** | An internet connection for the setup cells: the data and the course's files (6 MB) and the embedding model (about 470 MB, once). No account and no key: the LLM's answers were recorded. |
| **You should know** | One yes/no model per label, from [Multi-label messages](https://learning.nextia-ai.com/courses/text-routing/m07/multi-label-messages/); two levels of labels, from [Hierarchical labels](https://learning.nextia-ai.com/courses/text-routing/m07/hierarchical-labels/); rare classes, from [Imbalance and rare classes](https://learning.nextia-ai.com/courses/text-routing/m07/imbalance-and-rare-classes/). |
| **Data** | arXiv paper metadata, CC0 1.0 (public domain), a fixed sample of 12,000 papers. Details in `dataset.md` next to this notebook. |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, sentence-transformers 6.2.0 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/tag-research-papers-with-many-labels/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M10-L04-tag-research-papers-with-many-labels/tag-research-papers-with-many-labels-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import sentence_transformers; print("sentence-transformers", sentence_transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the data and the course's files

Run the next cell. It downloads into a folder `arxiv-papers`:

- the sample of 12,000 arXiv papers, `data/papers.csv.gz` (5 MB), and the names of the 78 categories;
- the course's files: the recorded LLM answers and the helper `routing.py`.

It checks each file's checksum. You should see `Ready: 5 files in arxiv-papers/`. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `arxiv-papers`, then run the cell again.

**Source and licence.** Paper metadata from [arXiv](https://arxiv.org), dedicated to the public domain under [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/), from the snapshot [`gfissore/arxiv-abstracts-2021`](https://huggingface.co/datasets/gfissore/arxiv-abstracts-2021) on Hugging Face. `dataset.md` lists how we made the sample and what we changed.

In [ ]:
# Setup: download the data and the case study's files into arxiv-papers/ and check each file.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "data/papers.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L04-tag-research-papers-with-many-labels/data/papers.csv.gz",
        "d56bc3a6550e1bc1210ba67080d271e4f27d57842bf498adbd27c840a62bbafb"
    ],
    "data/category_names.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L04-tag-research-papers-with-many-labels/data/category_names.csv",
        "b3474270d2e00d06df49025c6e26028412c10da6de3423734a73e20da9b6b6ea"
    ],
    "recordings/llm-zero-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L04-tag-research-papers-with-many-labels/recordings/llm-zero-shot.jsonl",
        "60a2ba30c29cead6c7a0ba97ea0753c9a14421741a3435af001b31c55474a7a0"
    ],
    "recordings/llm-few-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L04-tag-research-papers-with-many-labels/recordings/llm-few-shot.jsonl",
        "460edc87d863144af7ade94804c35bc25d72dcacaaee076cb419108fc2822deb"
    ],
    "routing.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/routing.py",
        "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c"
    ],
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("arxiv-papers").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder arxiv-papers and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in {PROJECT.name}/")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, PyTorch and the course helper as `rt`, and defines the check helpers. They tell you if your answer is right, without showing it:

- `expect(what, yours, expected)` compares one of your numbers with the expected one.
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the libraries, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
import json, time, torch
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the first checks

One row is one paper: its ID, title, abstract and categories. The next cell loads the sample and runs the first checks. Note `dtype={"id": str}`: arXiv IDs are text, not numbers.

> **Check.** You should see 12,000 papers in four areas, no missing values, and no category without a name.

In [ ]:
papers = pd.read_csv("data/papers.csv.gz", dtype={"id": str})   # arXiv IDs are text: 2104.02810 is not 2104.0281
names = pd.read_csv("data/category_names.csv")
NAME_OF = dict(zip(names.category, names.name))
papers["tags"] = papers.categories.str.split()          # the labels of each paper, as a list
papers["text"] = papers.title + ". " + papers.abstract
print(f"papers: {len(papers):,} | areas: {papers.area.value_counts().to_dict()}")
print(f"missing values: {papers[['title', 'abstract', 'categories']].isna().sum().sum()} | repeated IDs: "
      f"{papers.id.duplicated().sum()} | repeated abstracts: {papers.abstract.duplicated().sum()}")
words = papers.abstract.str.split().str.len()
print(f"words per abstract: median {words.median():.0f}, 90% have at most {words.quantile(0.9):.0f}, longest {words.max()}")
n_tags = papers.tags.str.len()
print("categories per paper:", n_tags.value_counts().sort_index().to_dict())
print("categories without a name:", sorted(set(papers.tags.explode()) - set(NAME_OF)))

arXiv gives four categories two names. The next cell checks that both names always come together. We merged each pair into one name when we made the sample, so a model does not get two "free" labels.

In [ ]:
# arXiv gives some categories two names (an alias pair), and lists both on every paper.
raw = papers.raw_categories.str.split()
for a, b in [("math.ST", "stat.TH"), ("cs.IT", "math.IT"), ("math.NA", "cs.NA"), ("eess.SY", "cs.SY")]:
    has_a, has_b = raw.map(lambda c: a in c), raw.map(lambda c: b in c)
    print(f"{a} and {b}: {has_a.sum()} and {has_b.sum()} papers; always together: {(has_a == has_b).all()}")
outside = (raw.str.len() > papers.tags.str.len()).mean()
print(f"papers that also had a category outside the four areas (dropped): {outside:.1%}")

## 2. The labels and the long tail

The next cell makes the three parts and counts the papers of each category in the training part. A category with fewer than 20 training papers is **too rare** to learn here. We keep it in the data, but the model never suggests it.

> **Check.** You should see 70 categories that we learn and 8 that are too rare.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
train = papers[papers.split == "train"].reset_index(drop=True)
validation = papers[papers.split == "validation"].reset_index(drop=True)
test = papers[papers.split == "test"].reset_index(drop=True)
print(f"train {len(train):,} | validation {len(validation):,} | test {len(test):,} (used once, at the end)")
per_label = train.tags.explode().value_counts()
MIN_PAPERS = 20                                  # a model needs examples: fewer than 20 is too rare to learn here
LABELS = list(per_label[per_label >= MIN_PAPERS].index)
TOO_RARE = list(per_label[per_label < MIN_PAPERS].index)
RARE = list(per_label[(per_label >= MIN_PAPERS) & (per_label < 50)].index)   # the long tail that we can learn
mlb = MultiLabelBinarizer(classes=LABELS)
Y = {name: mlb.fit_transform(part.tags.map(lambda t: [x for x in t if x in LABELS]))
     for name, part in [("train", train), ("validation", validation), ("test", test)]}
print(f"categories: {len(per_label)} in train | learned: {len(LABELS)} | too rare (< {MIN_PAPERS} papers): {len(TOO_RARE)}"
      f" | rare (20 to 49 papers): {len(RARE)}")
print("most common:", per_label.head(5).to_dict())
print("too rare:", per_label[TOO_RARE].to_dict())

> **Your turn.** What share of the training papers have more than one category? Use `train.tags`.

> **Predict.** About 10%, about 30% or about 50%?

In [ ]:
multi_share = ...   # the share of training papers with more than one category, rounded to 3 decimals

In [ ]:
expect('multi_share', multi_share, 0.514, tolerance=0.002)

## 3. The cost table and the helpers

An assistant suggests categories, and a person confirms them. The costs are **assumptions** for this case study:

- a right category that the assistant does not suggest: **1.00 dollar** (the person must find it in a list of 70, or the paper misses those readers);
- a wrong suggestion: **0.20 dollars** (the person reads it and removes it);
- no suggestion at all: **1.50 dollars**, the course's cost of a review (the person chooses the categories alone).

`suggest()` turns probabilities into yes/no suggestions with a threshold. `score()` returns one row of the comparison table. Run the cell.

In [ ]:
# The case study's cost table: assumptions for an assistant that suggests categories to a person.
from sklearn.metrics import f1_score
MISSED = 1.00                   # a right category that is not suggested: the person must find it in the list, or forget it
WRONG = 0.20                    # a wrong suggestion: the person reads it and removes it (about 10 seconds)
MANUAL = rt.COSTS["review"]     # no suggestion at all: the person chooses the categories alone (1.50, about a minute)
CPU_HOUR = 0.05                 # dollars per CPU-hour (the course's assumption)
rare_cols = [LABELS.index(x) for x in RARE]

def suggest(P, threshold, at_least_one=False, top=5):
    """Yes/no for each paper and category: the probability reaches its threshold (one number or one per
    category). At most `top` suggestions per paper. With at_least_one, a paper that passes no threshold
    gets its most likely category; without it, the paper goes to the person with no suggestion."""
    P = np.asarray(P)
    yes = P >= np.asarray(threshold)
    rank = np.argsort(-P, axis=1).argsort(axis=1)        # 0 for the most likely category of each paper
    yes &= rank < top
    if at_least_one:
        yes |= rank == 0
    return yes

def truth_of(part):
    """Yes/no for each paper of a part and each learned category: the labels."""
    return mlb.transform(part.tags.map(lambda t: [x for x in t if x in LABELS]))

def score(name, part, yes, run_per_1000=0.0, ms=None):
    """One row of the comparison table for a part (validation, test or a sample) and the yes/no suggestions.
    A paper with no suggestion goes to the person (MANUAL)."""
    truth, yes = truth_of(part), np.asarray(yes, bool)
    rare_tags = part.tags.map(lambda t: len([x for x in t if x in TOO_RARE])).to_numpy()
    empty = ~yes.any(axis=1)
    missed = ((truth == 1) & ~yes).sum(axis=1) + rare_tags      # the too-rare categories are always missed
    wrong = (yes & (truth == 0)).sum(axis=1)
    cost = np.where(empty, MANUAL, missed * MISSED + wrong * WRONG)
    rare_true = truth[:, rare_cols] == 1
    return {"method": name, "exact_match": round(float(((truth == yes).all(axis=1) & (rare_tags == 0)).mean()), 3),
            "micro_f1": round(float(f1_score(truth, yes, average="micro", zero_division=0)), 3),
            "macro_f1": round(float(f1_score(truth, yes, average="macro", zero_division=0)), 3),
            "rare_recall": round(float((yes[:, rare_cols] & rare_true).sum() / rare_true.sum()), 3),
            "no_tag_share": round(float(empty.mean()), 3),
            "tags_per_paper": round(float(yes.sum(axis=1).mean()), 2),
            "mistakes_per_1000": round(float(cost.mean() * 1000), 1), "run_per_1000": round(run_per_1000, 3),
            "total_per_1000": round(float(cost.mean() * 1000 + run_per_1000), 1),
            "ms_per_paper": None if ms is None else round(ms, 2)}

def table(rows):
    """The rows as a table, cheapest total first."""
    return pd.DataFrame(rows).sort_values("total_per_1000").reset_index(drop=True)

def cpu_cost(ms):
    return ms / 1000 / 3600 * CPU_HOUR * 1000   # dollars per 1,000 papers at `ms` milliseconds each

def ms_per_paper(predict, texts, n=200):
    t0 = time.perf_counter()
    for t in texts[:n]:
        predict([t])
    return (time.perf_counter() - t0) / n * 1000

## 4. The baseline: one yes/no model per category

TF-IDF and one logistic regression per category (one-vs-rest), as in the multi-label lesson. With 0.50 as the threshold for every category:

> **Check.** About 49.4% of the validation papers get no suggestion at all, and the cost is about 1138.7 dollars per 1,000 papers.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline

def tfidf_ovr():
    """TF-IDF of words and word pairs, and one yes/no logistic regression per category (one-vs-rest)."""
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True, strip_accents="unicode")
    return make_pipeline(words, OneVsRestClassifier(LogisticRegression(C=10, max_iter=2000), n_jobs=1))

t0 = time.perf_counter()
tfidf = tfidf_ovr().fit(train.text, Y["train"])          # 71 yes/no models
print(f"fitted {len(LABELS)} yes/no models in {time.perf_counter() - t0:.0f} s")
P_tfidf = {"validation": tfidf.predict_proba(validation.text)}
tfidf_ms = ms_per_paper(tfidf.predict_proba, list(validation.text))
rows_val = [score("everything to a person", validation, np.zeros_like(Y["validation"], bool)),
            score("TF-IDF, 0.50 for every category", validation, suggest(P_tfidf["validation"], 0.5),
                  cpu_cost(tfidf_ms), tfidf_ms)]
table(rows_val)

Look at one paper and its scores. The scores of the 70 models do not add up to 1: each model answers its own question.

In [ ]:
# One validation paper: its true categories and the scores of the 71 yes/no models.
i = int(np.flatnonzero(validation.id == "2006.00198")[0])
paper = validation.iloc[i]
print(paper.title)
print("true categories:", paper.categories)
top = pd.Series(P_tfidf["validation"][i], index=LABELS).sort_values(ascending=False).head(6).round(3)
print(top.to_string())

## 5. A threshold from the cost table

> **Your turn.** If `p` is the chance that a category is right, suggesting it costs `(1 - p) * WRONG` on average, and leaving it out costs `p * MISSED`. At which `p` are the two costs equal?

In [ ]:
# Suggest a category when the expected cost of suggesting it is lower than the expected cost of leaving it out.
# If p is the chance that the category is right: suggesting costs (1 - p) * WRONG, leaving out costs p * MISSED.
break_even = ...   # the probability where both costs are equal, rounded to 3 decimals

In [ ]:
expect('break_even', break_even, 0.167, tolerance=0.002)

## 6. Choose the thresholds on validation

The next cell tries every threshold from 0.05 to 0.90 for all categories, then a threshold per category, and the rule "at least one suggestion".

> **Predict.** The break-even probability was 0.167. Will the cheapest threshold be higher or lower?

> **Check.** The cheapest single threshold is 0.07.

In [ ]:
# Choose the thresholds on the validation papers, by cost.
grid = np.round(np.arange(0.05, 0.91, 0.01), 2)
curve = pd.DataFrame([{"threshold": t, **score("TF-IDF", validation, suggest(P_tfidf["validation"], t))}
                      for t in grid])
best = curve.loc[curve.total_per_1000.idxmin()]
GLOBAL = float(best.threshold)
print(f"one threshold for every category: lowest cost at {GLOBAL:.2f}, {best.total_per_1000:.1f} dollars per 1,000, "
      f"{best.no_tag_share:.1%} to a person (at 0.50: {curve.set_index('threshold').total_per_1000[0.5]:.1f})")

def per_label_thresholds(P, truth, min_positives=10, fallback=None):
    """For each category, the threshold with the lowest cost of misses and wrong suggestions on these papers.
    A category with fewer than min_positives right papers here keeps the fallback threshold (too few to choose)."""
    out = []
    for j in range(truth.shape[1]):
        pos = truth[:, j] == 1
        if pos.sum() < min_positives:
            out.append(fallback)
            continue
        costs = [MISSED * (pos & (P[:, j] < t)).sum() + WRONG * (~pos & (P[:, j] >= t)).sum() for t in grid]
        out.append(float(grid[int(np.argmin(costs))]))
    return np.array(out)

th_tfidf = per_label_thresholds(P_tfidf["validation"], Y["validation"], fallback=GLOBAL)
print(f"thresholds per category: from {th_tfidf.min():.2f} to {th_tfidf.max():.2f}, median {np.median(th_tfidf):.2f}; "
      f"{int((th_tfidf == GLOBAL).sum())} categories keep {GLOBAL:.2f}")
SPEC = {f"TF-IDF, {GLOBAL:.2f} for every category": ("TF-IDF", GLOBAL, False),   # name: method, threshold, at least one
        "TF-IDF, a threshold per category": ("TF-IDF", th_tfidf, False),
        f"TF-IDF, {GLOBAL:.2f}, at least one": ("TF-IDF", GLOBAL, True)}
rows_val += [score(f"TF-IDF, {GLOBAL:.2f} for every category", validation, suggest(P_tfidf["validation"], GLOBAL),
                   cpu_cost(tfidf_ms), tfidf_ms),
             score("TF-IDF, a threshold per category", validation, suggest(P_tfidf["validation"], th_tfidf),
                   cpu_cost(tfidf_ms), tfidf_ms),
             score(f"TF-IDF, {GLOBAL:.2f}, at least one", validation,
                   suggest(P_tfidf["validation"], GLOBAL, at_least_one=True), cpu_cost(tfidf_ms), tfidf_ms)]
shown = curve.threshold.isin([0.05, 0.1, GLOBAL, 0.2, 0.3, 0.5, 0.7, 0.9])
print(curve.loc[shown, ["threshold", "micro_f1", "macro_f1", "no_tag_share", "tags_per_paper", "total_per_1000"]].to_string(index=False))
table(rows_val)

> **Your turn.** The person can tag at most 2% of the papers alone (no suggestion). Find the cheapest single threshold under that limit, from `curve`.

In [ ]:
# The person can tag at most 2% of the papers alone. Find the cheapest single threshold under that limit.
threshold_2 = ...   # a value from curve.threshold

In [ ]:
expect('threshold_2', threshold_2, 0.05)

## 7. Two levels: area, then category

Categories have two levels: the area (`cs`, `math`, `stat`, `eess`) and the category inside it (`cs.LG`). The next cell compares the two levels, and treats the area of the primary category as a routing problem with `routing.py`.

> **Check.** The area level is much easier than the category level. A model for the 4 areas reaches an accuracy of about 0.842.

In [ ]:
# Two levels: the top-level area (cs, math, stat, eess), then the categories inside it.
AREAS = ["cs", "math", "stat", "eess"]
area_of = np.array([x.split(".")[0] for x in LABELS])
def to_areas(yes):
    """From yes/no per category to yes/no per area: an area is yes when one of its categories is yes."""
    return np.stack([np.asarray(yes)[:, area_of == a].any(axis=1) for a in AREAS], axis=1)

yes_val = suggest(P_tfidf["validation"], GLOBAL)
area_truth = np.stack([validation.tags.map(lambda t: any(x.startswith(a + ".") for x in t)) for a in AREAS], axis=1)
print(f"micro-F1 by category: {f1_score(Y['validation'], yes_val, average='micro'):.3f} | by area: "
      f"{f1_score(area_truth, to_areas(yes_val), average='micro'):.3f}")
wrong = yes_val & (Y["validation"] == 0)
own = np.array([[area_of[j] in {x.split('.')[0] for x in validation.tags[i]} for j in range(len(LABELS))]
                for i in range(len(validation))])
print(f"wrong suggestions: {wrong.sum()}, of which in an area that the paper has: {(wrong & own).sum()} "
      f"({(wrong & own).sum() / wrong.sum():.0%})")

# The first level as a routing problem: send each paper to the moderators of its primary area (routing.py).
rt.ROUTES = AREAS
def routes_of(part):
    return pd.DataFrame({"route": part.area.values, "topic": part.primary.values, "priority": "normal",
                         "language": "en"}, index=part.index)
area_from_tags = area_of[P_tfidf["validation"].argmax(axis=1)]           # the area of the most likely category
area_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True),
                           LogisticRegression(C=10, max_iter=2000)).fit(train.text, train.area)
areas_val = pd.DataFrame([rt.score("area of the top category", routes_of(validation), area_from_tags),
                          rt.score("a model for the 4 areas", routes_of(validation), area_model.predict(validation.text))])
areas_val[["method", "accuracy", "macro_f1", "total_per_1000"]]

## 8. Frozen embeddings, and both together

The course's older embedding model turns each title and abstract into 384 numbers. It reads only the first 128 word pieces. The cell embeds all 12,000 papers: about a minute and a half on our laptop, longer on Colab, plus the model download the first time. Then it tries the mean of the TF-IDF and embedding probabilities, and chooses the cheapest router on validation.

> **Check.** The cheapest router on validation is `TF-IDF, a threshold per category`.

In [ ]:
from sentence_transformers import SentenceTransformer
torch.set_num_threads(4)
# The course's older embedding model, frozen. It reads the first 128 word pieces of each title and abstract.
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                               revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42", device="cpu")
t0 = time.perf_counter()
E = {name: embedder.encode(list(part.text), batch_size=64, normalize_embeddings=True)
     for name, part in [("train", train), ("validation", validation), ("test", test)]}
print(f"{sum(len(v) for v in E.values()):,} papers embedded in {time.perf_counter() - t0:.0f} s "
      f"(max length {embedder.max_seq_length} word pieces)")
frozen = OneVsRestClassifier(LogisticRegression(C=10, max_iter=2000), n_jobs=1).fit(E["train"], Y["train"])
P_frozen = {"validation": frozen.predict_proba(E["validation"])}
frozen_ms = ms_per_paper(lambda t: frozen.predict_proba(embedder.encode(t, normalize_embeddings=True)),
                         list(validation.text), n=100)
P_both = {"validation": (P_tfidf["validation"] + P_frozen["validation"]) / 2}   # the mean of the two probabilities
both_ms = tfidf_ms + frozen_ms
METHODS = {"TF-IDF": (P_tfidf, tfidf_ms), "embeddings": (P_frozen, frozen_ms), "TF-IDF + embeddings": (P_both, both_ms)}
G, TH = {"TF-IDF": GLOBAL}, {"TF-IDF": th_tfidf}
for key in ["embeddings", "TF-IDF + embeddings"]:
    P, ms = METHODS[key]
    c = [score(key, validation, suggest(P["validation"], t))["total_per_1000"] for t in grid]
    G[key] = float(grid[int(np.argmin(c))])
    TH[key] = per_label_thresholds(P["validation"], Y["validation"], fallback=G[key])
    SPEC[f"{key}, {G[key]:.2f} for every category"] = (key, G[key], False)
    SPEC[f"{key}, a threshold per category"] = (key, TH[key], False)
    rows_val += [score(f"{key}, {G[key]:.2f} for every category", validation, suggest(P["validation"], G[key]), cpu_cost(ms), ms),
                 score(f"{key}, a threshold per category", validation, suggest(P["validation"], TH[key]), cpu_cost(ms), ms)]
CHOSEN = table(rows_val).method[0]          # the cheapest on the validation papers
print("chosen on validation:", CHOSEN)
table(rows_val)

## 9. The hosted LLM, recorded

A hosted LLM (Azure chat-small) chose categories for 400 validation and 400 test papers, with a confidence for each. The zero-shot prompt has the list of the 70 categories with their names. The few-shot prompt adds the titles and categories of the 5 most similar training papers. The next cell shows the few-shot prompt for one paper. Its answers were recorded once: you replay them, with no key.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# The prompt of the recorded LLM answers. The few-shot prompt adds the titles of the 5 most similar training papers.
label_list = "\n".join(f"- {x}: {NAME_OF[x]}" for x in sorted(LABELS))
SYSTEM = ("You suggest arXiv categories for a new paper, from its title and abstract. Choose every category that "
          f"fits, from this list only:\n{label_list}\n"
          "Give each chosen category a confidence from 0 to 1. Most papers have 1 to 3 categories.")
picker = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True)
picker_x = picker.fit_transform(train.text)

def examples_for(text, k=5):
    """The k training papers with the closest TF-IDF vectors, as (title, categories)."""
    sims = (picker_x @ picker.transform([text]).T).toarray().ravel()
    return [(train.title[i], " ".join(x for x in train.tags[i] if x in LABELS)) for i in sims.argsort()[::-1][:k]]

def user_prompt(text, mode):
    if mode == "zero-shot":
        return f"Paper:\n{text}"
    shown = "\n".join(f"- {t!r} -> {c}" for t, c in examples_for(text))
    return f"Similar papers and their categories:\n{shown}\n\nPaper:\n{text}"

# The samples with recorded answers: 400 validation and 400 test papers.
validation_sample = validation.sample(400, random_state=36).sort_index()
test_sample = test.sample(400, random_state=36).sort_index()
print(user_prompt(test_sample.text.iloc[0][:300] + " ...", "few-shot"))

The LLM was recorded for a sample only, so the next cell compares the chosen router with the LLM on that same sample. The LLM's confidence threshold is chosen on the validation sample, by cost.

In [ ]:
def recorded(mode):
    """The recorded answers of the hosted LLM (chat-small), one line per paper."""
    rows = [json.loads(x) for x in open(f"recordings/llm-{mode}.jsonl", encoding="utf-8")]
    return pd.DataFrame(rows).set_index("id")

def llm_proba(answers, part):
    """The LLM's confidence for each category (0 when it did not choose the category)."""
    P = np.zeros((len(part), len(LABELS)))
    for i, pid in enumerate(part.id):
        for c in answers.loc[pid, "categories"]:
            if c["category"] in LABELS:
                P[i, LABELS.index(c["category"])] = c["confidence"]
    return P

PRICE_IN, PRICE_OUT = 0.10, 0.50   # chat-small, dollars per 1 million tokens (the course's price table)
LLM = {}
for mode in ["zero-shot", "few-shot"]:
    a = recorded(mode)
    LLM[mode] = {"answers": a, "run": (a.tokens_in.mean() * PRICE_IN + a.tokens_out.mean() * PRICE_OUT) / 1e6 * 1000,
                 "ms": a.latency_s.median() * 1000}
    print(f"{mode}: {len(a)} answers, all valid: {a.valid.all()} | tokens in {a.tokens_in.mean():.0f}, out "
          f"{a.tokens_out.mean():.0f} | {LLM[mode]['run']:.3f} dollars per 1,000 | median {LLM[mode]['ms']:.0f} ms")

vs = validation_sample.index
key, th, one = SPEC[CHOSEN]
rows_sample = [score(f"TF-IDF, {GLOBAL:.2f} for every category", validation_sample, suggest(P_tfidf["validation"][vs], GLOBAL),
                     cpu_cost(tfidf_ms)),
               score(CHOSEN, validation_sample, suggest(METHODS[key][0]["validation"][vs], th, one), cpu_cost(METHODS[key][1]))]
LLM_TH = {}
for mode in ["zero-shot", "few-shot"]:
    P = llm_proba(LLM[mode]["answers"], validation_sample)
    c = {t: score("LLM", validation_sample, suggest(P, t))["total_per_1000"] for t in [0.01, 0.3, 0.5, 0.7, 0.8, 0.9]}
    LLM_TH[mode] = min(c, key=c.get)        # the LLM's confidence threshold, chosen on the validation sample by cost
    rows_sample.append(score(f"LLM {mode}, confidence >= {LLM_TH[mode]} (recorded)", validation_sample,
                             suggest(P, LLM_TH[mode]), LLM[mode]["run"], LLM[mode]["ms"]))
table(rows_sample)

## 10. The test, once

Now the test papers, for every router. Nothing changes after this cell.

> **Check.** The chosen router costs about 754.9 dollars per 1,000 papers, with a micro-F1 of about 0.592, and sends 5.5% of the papers to the person with no suggestion.

In [ ]:
# The test papers, once. Every choice was made on the validation papers; nothing changes after this cell.
P_tfidf["test"] = tfidf.predict_proba(test.text)
P_frozen["test"] = frozen.predict_proba(E["test"])
P_both["test"] = (P_tfidf["test"] + P_frozen["test"]) / 2
rows_test = [score("everything to a person", test, np.zeros_like(Y["test"], bool)),
             score("TF-IDF, 0.50 for every category", test, suggest(P_tfidf["test"], 0.5), cpu_cost(tfidf_ms), tfidf_ms)]
for name, (key, th, one) in SPEC.items():
    P, ms = METHODS[key]
    rows_test.append(score(name, test, suggest(P["test"], th, one), cpu_cost(ms), ms))
table(rows_test)

And on the test sample with the recorded LLM answers:

In [ ]:
# The LLM was recorded for a sample of the test papers: compare on that same sample.
ts = test_sample.index
key, th, one = SPEC[CHOSEN]
rows_test_sample = [score(f"TF-IDF, {GLOBAL:.2f} for every category", test_sample, suggest(P_tfidf["test"][ts], GLOBAL),
                          cpu_cost(tfidf_ms)),
                    score(CHOSEN, test_sample, suggest(METHODS[key][0]["test"][ts], th, one), cpu_cost(METHODS[key][1]))]
for mode in ["zero-shot", "few-shot"]:
    P = llm_proba(LLM[mode]["answers"], test_sample)
    rows_test_sample.append(score(f"LLM {mode}, confidence >= {LLM_TH[mode]} (recorded)", test_sample,
                                  suggest(P, LLM_TH[mode]), LLM[mode]["run"], LLM[mode]["ms"]))
table(rows_test_sample)

## 11. Where the router still fails

The next cell gives the F1 score of each category on the test papers, by how many training papers it had, and the categories that are missed most often.

In [ ]:
# Where the chosen router still fails: F1 per category on the test papers, against the number of training papers.
from sklearn.metrics import precision_recall_fscore_support
key, th, one = SPEC[CHOSEN]
yes_test = suggest(METHODS[key][0]["test"], th, one)
prec, rec, f1, support = precision_recall_fscore_support(Y["test"], yes_test, zero_division=0)
by_label = pd.DataFrame({"category": LABELS, "train_papers": per_label[LABELS].values, "test_papers": support,
                         "precision": prec.round(3), "recall": rec.round(3), "f1": f1.round(3)})
right = (yes_test & (Y["test"] == 1)).sum()
print(f"suggestions: {yes_test.sum()}, right: {right} ({right / yes_test.sum():.0%}) | right categories found: "
      f"{right} of {Y['test'].sum()} ({right / Y['test'].sum():.0%})")
for lo, hi in [(20, 50), (50, 100), (100, 300), (300, 2000)]:
    g = by_label[(by_label.train_papers >= lo) & (by_label.train_papers < hi)]
    print(f"{lo:>4} to {hi - 1:>4} training papers: {len(g):>2} categories, mean F1 {g.f1.mean():.3f}")
pairs = []
for j, lab in enumerate(LABELS):
    miss = (Y["test"][:, j] == 1) & ~yes_test[:, j]
    if miss.sum():
        given = yes_test[miss].sum(axis=0)
        k = int(given.argmax())
        if given[k]:
            pairs.append({"category": lab, "missed": int(miss.sum()), "suggested_instead": LABELS[k], "times": int(given[k])})
pairs = pd.DataFrame(pairs).sort_values("missed", ascending=False).reset_index(drop=True)
by_label.sort_values("f1").head(8)

In [ ]:
pairs.head(8)

## 12. New abstracts

We wrote three short abstracts, in the style of arXiv. The chosen router suggests categories for each one.

In [ ]:
# New input, with the chosen router: three short abstracts that we wrote for this case study (not from arXiv).
new = pd.DataFrame({"text": [
    "Counting lattice paths with forbidden steps. We count lattice paths in the plane that avoid a fixed set of "
    "steps, and we give a closed formula with binomial coefficients and a bijection with Dyck words.",
    "A small transformer for detecting faults in power grids. We train a transformer on voltage signals from "
    "sensors in a power network and detect faults in real time. A controller then isolates the damaged line.",
    "Confidence intervals after model selection. We study the coverage of confidence intervals when the variables "
    "of a regression model were chosen from the same data, and propose a bootstrap correction."]})
key, th, one = SPEC[CHOSEN]
P_new = {"TF-IDF": lambda: tfidf.predict_proba(new.text),
         "embeddings": lambda: frozen.predict_proba(embedder.encode(list(new.text), normalize_embeddings=True)),
         "TF-IDF + embeddings": lambda: (tfidf.predict_proba(new.text) + frozen.predict_proba(
             embedder.encode(list(new.text), normalize_embeddings=True))) / 2}[key]()
yes_new = suggest(P_new, th, one)
new["suggested"] = [", ".join(f"{LABELS[j]} {P_new[i, j]:.2f}" for j in np.argsort(-P_new[i]) if yes_new[i, j]) or "to a person"
                    for i in range(len(new))]
new["title"] = new.text.str.split(".").str[0]
new[["title", "suggested"]]

Read each row. Would you accept these suggestions? Which category would you add or remove?

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/tag-research-papers-with-many-labels/) explains each step, the costs, the recommendation and the limits, and how to make this case study a portfolio piece. Its knowledge checks count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/text-routing/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m10/tag-research-papers-with-many-labels/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). arXiv paper metadata, CC0 1.0. The categories are the authors' own choices.